In [1]:
"""
01 - Load FlavorGraph assets, scan node availability for MoBai-relevant ingredients.

Outputs:
- data/mobai_target_nodes.csv: curated mapping MoBai flavor -> FlavorGraph node + strategy
- Prints summary stats to stdout
"""
import pickle
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path("__file__" in globals() and __file__ or "notebooks/dummy.py").resolve().parents[1] if "__file__" in globals() else Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

In [3]:
# Load embeddings
with open(DATA / "flavorgraph_embeddings.pickle", "rb") as f:
    emb = pickle.load(f)
print(f"Loaded embeddings: {len(emb)} nodes, dim={next(iter(emb.values())).shape[0]}")

Loaded embeddings: 8297 nodes, dim=300


/tmp/ipykernel_136112/1433871426.py:3: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  emb = pickle.load(f)


In [4]:
# Load node metadata
nodes = pd.read_csv(DATA / "nodes_191120.csv")
print(f"Loaded node list: {len(nodes)} rows, columns={list(nodes.columns)}")
print(f"Node types: {nodes['node_type'].value_counts().to_dict()}")
print(f"Hub status: {nodes['is_hub'].value_counts().to_dict()}")

Loaded node list: 8298 rows, columns=['node_id', 'name', 'id', 'node_type', 'is_hub']
Node types: {'ingredient': 6653, 'compound': 1645}
Hub status: {'no_hub': 6237, 'food': 1561, 'hub': 416, 'drug': 84}


In [5]:
# Build lookup: name -> node_id (note: node_id in CSV is int, in pickle is str)
nodes['node_id_str'] = nodes['node_id'].astype(str)
name_to_id = dict(zip(nodes['name'], nodes['node_id_str']))

In [6]:
# Search for MoBai-relevant flavors (from NLP Q1 + product spec)
mobai_search_terms = {
    # NLP Q1 chart flavors (13 total)
    "mango": ["mango", "fresh_mango", "dried_mango"],
    "vanilla": ["vanilla", "vanilla_extract", "vanilla_bean"],
    "caramel": ["caramel", "caramel_sauce"],
    "coconut": ["coconut", "coconut_milk", "coconut_cream", "coconut_water"],
    "chocolate": ["chocolate", "dark_chocolate", "milk_chocolate"],
    "milk_tea": ["milk_tea", "bubble_tea"],  # likely missing
    "oat": ["oat", "oats", "rolled_oats", "oatmeal"],
    "coffee": ["coffee", "brewed_coffee", "instant_coffee"],
    "banana": ["banana", "ripe_banana"],
    "matcha": ["matcha", "green_tea_powder"],
    "peach": ["peach", "fresh_peach", "canned_peach", "dried_peach"],
    "honey": ["honey"],
    "strawberry": ["strawberry", "fresh_strawberry"],
    # MoBai product spec
    "jasmine": ["jasmine", "jasmine_tea", "jasmine_flower"],
    "black_tea": ["black_tea", "brewed_black_tea"],
    "oolong": ["oolong", "oolong_tea"],
    "milk": ["milk", "whole_milk", "skim_milk", "low_fat_milk"],
    "yogurt": ["yogurt", "plain_yogurt", "greek_yogurt"],
    "tea": ["tea", "brewed_tea", "green_tea", "brewed_green_tea"],
    # Compound proxies for jasmine
    "benzyl_acetate": ["benzyl_acetate"],
    "linalool": ["linalool"],
    "methyl_jasmonate": ["methyl_jasmonate"],
    # Variant C candidates (per MoBai Section 4.1)
    "osmanthus": ["osmanthus"],
    "white_peach": ["white_peach"],
    "lychee": ["lychee", "lychee_fruit"],
    "passionfruit": ["passionfruit", "passion_fruit"],
}

In [7]:
results = []
for mobai_term, candidates in mobai_search_terms.items():
    found = []
    for c in candidates:
        # Exact match
        if c in name_to_id and name_to_id[c] in emb:
            found.append((c, name_to_id[c], "exact"))
        # Substring match
        else:
            matches = nodes[nodes['name'].str.contains(c, case=False, na=False)]
            for _, row in matches.head(5).iterrows():
                nid = str(row['node_id'])
                if nid in emb:
                    found.append((row['name'], nid, "substring"))
    results.append({
        "mobai_term": mobai_term,
        "matches": "; ".join([f"{n}({i})" for n, i, _ in found[:5]]) if found else "NONE",
        "count": len(found),
        "status": "FOUND" if found else "MISSING",
    })

In [8]:
results_df = pd.DataFrame(results)
print("\n=== MoBai Node Availability ===")
for _, row in results_df.iterrows():
    print(f"  [{row['status']:7s}] {row['mobai_term']:20s} ({row['count']} matches): {row['matches'][:120]}")


=== MoBai Node Availability ===
  [FOUND  ] mango                (3 matches): mango(4052); fresh_mango(2528); dried_mango(2021)
  [FOUND  ] vanilla              (3 matches): vanilla(6631); vanilla_extract(6644); vanilla_bean(6633)
  [FOUND  ] caramel              (2 matches): caramel(977); caramel_sauce(982)
  [FOUND  ] coconut              (5 matches): coconut(1430); coconut_milk(1444); instant_coconut_cream_pudding_mix(3408); unsweetened_coconut_cream(6606); coconut_wat
  [FOUND  ] chocolate            (3 matches): chocolate(1266); dark_chocolate(1810); milk_chocolate(4205)
  [MISSING] milk_tea             (0 matches): NONE
  [FOUND  ] oat                  (3 matches): oat(4444); honey_bunches_of_oats_cereal(3293); oatmeal(4447)
  [FOUND  ] coffee               (3 matches): coffee(1458); brewed_coffee(693); instant_coffee(3409)
  [FOUND  ] banana               (1 matches): banana(302)
  [FOUND  ] matcha               (2 matches): matcha_green_tea_powder(4107); green_tea_powder(3038)

In [9]:
# Build canonical mapping: choose primary node per MoBai term
canonical_map = {
    # Direct hits (verified against actual node names)
    "mango":        ("fresh_mango", "direct"),
    "vanilla":      ("vanilla", "direct"),
    "caramel":      ("caramel", "direct"),
    "coconut":      ("coconut", "direct"),
    "chocolate":    ("chocolate", "direct"),
    "oat":          ("oat", "direct"),
    "coffee":       ("coffee", "direct"),
    "banana":       ("banana", "direct"),
    "matcha":       ("matcha_green_tea_powder", "direct"),
    "peach":        ("canned_peach", "direct"),
    "honey":        ("honey", "direct"),
    "strawberry":   ("strawberry", "direct"),
    "black_tea":    ("black_tea", "direct"),
    "milk":         ("milk", "direct"),
    "yogurt":       ("yogurt", "direct"),
    "jasmine":      ("jasmine_tea", "direct"),                    # FOUND in full data
    # Compound proxies
    "milk_tea":     ("black_tea+milk", "compound_proxy"),         # not a single node, average
    # Additional jasmine aromatic compounds (alternative for sensitivity check)
    "jasmine_compound_BA": ("Benzyl_Acetate", "compound_proxy"),  # primary jasmine ester
    "jasmine_compound_lin": ("Linalool", "compound_proxy"),
    "jasmine_compound_lactone": ("Jasmine_lactone", "compound_proxy"),
    # Variant C candidates (verified present)
    "white_peach":  ("white_peach", "variant_c"),
    "lychee":       ("lychee", "variant_c"),
    "passionfruit": ("passion_fruit", "variant_c"),
    # Confirmed missing
    "oolong":       (None, "missing"),
    "osmanthus":    (None, "missing"),
}

In [10]:
# Verify each canonical mapping exists
out_rows = []
for term, (target, strategy) in canonical_map.items():
    if target is None:
        out_rows.append({"mobai_term": term, "flavorgraph_node": "", "node_id": "", "strategy": strategy, "in_embeddings": False})
        continue
    if "+" in target:
        # Compound proxy - verify each part
        parts = target.split("+")
        all_present = all(name_to_id.get(p) in emb for p in parts)
        out_rows.append({
            "mobai_term": term, "flavorgraph_node": target, "node_id": "+".join(name_to_id.get(p, "?") for p in parts),
            "strategy": strategy, "in_embeddings": all_present,
        })
    else:
        nid = name_to_id.get(target)
        in_emb = nid in emb if nid else False
        out_rows.append({
            "mobai_term": term, "flavorgraph_node": target, "node_id": nid or "",
            "strategy": strategy, "in_embeddings": in_emb,
        })

In [11]:
target_df = pd.DataFrame(out_rows)
target_df.to_csv(DATA / "mobai_target_nodes.csv", index=False)
print(f"\n=== Saved canonical mapping ===")
print(target_df.to_string(index=False))
print(f"\nWritten to: {DATA / 'mobai_target_nodes.csv'}")


=== Saved canonical mapping ===
              mobai_term        flavorgraph_node  node_id       strategy  in_embeddings
                   mango             fresh_mango     2528         direct           True
                 vanilla                 vanilla     6631         direct           True
                 caramel                 caramel      977         direct           True
                 coconut                 coconut     1430         direct           True
               chocolate               chocolate     1266         direct           True
                     oat                     oat     4444         direct           True
                  coffee                  coffee     1458         direct           True
                  banana                  banana      302         direct           True
                  matcha matcha_green_tea_powder     4107         direct           True
                   peach            canned_peach      928         direct           True